# Tesla vs GameStop: Share Price and Revenue Analysis
**Author:** Damaris Barasa | Python, pandas, yfinance, BeautifulSoup, Plotly

This project began as the final assignment of the IBM *Python Project for Data Science* course. The course covered extracting stock prices and scraping quarterly revenue. I extended it with my own analysis of risk, return and the link between revenue and share price.

**Questions I answer**
1. How did Tesla's and GameStop's share prices and revenue evolve?
2. Which stock gave higher returns, and which was riskier?
3. Does quarterly revenue move together with the share price?

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Load and clean the data
Stock prices come from Yahoo Finance. Quarterly revenue is scraped from static HTML pages provided by the course, which end in 2021, so I compare both companies up to the same cut-off date.

In [2]:
REVENUE_URLS = {
    "Tesla": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm",
    "GameStop": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html",
}
CUTOFF = pd.Timestamp("2021-06-14")   # revenue pages end in 2021

def get_stock(ticker):
    df = yf.Ticker(ticker).history(period="max").reset_index()
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
    df = df[["Date", "Close", "Volume"]]
    return df[df["Date"] <= CUTOFF].reset_index(drop=True)

def get_revenue(url):
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    rows = []
    for tr in soup.find_all("tbody")[1].find_all("tr"):      # table 1 = quarterly revenue
        td = tr.find_all("td")
        if len(td) >= 2:
            rows.append((td[0].text, td[1].text))
    df = pd.DataFrame(rows, columns=["Date", "Revenue"])
    df["Revenue"] = df["Revenue"].str.replace(r"[,$]", "", regex=True)
    df = df[df["Revenue"] != ""].dropna()
    df["Revenue"] = df["Revenue"].astype(float)              # US$ millions
    df["Date"] = pd.to_datetime(df["Date"])
    return df.sort_values("Date").reset_index(drop=True)

In [3]:
tesla_data = get_stock("TSLA")
gme_data = get_stock("GME")
tesla_revenue = get_revenue(REVENUE_URLS["Tesla"])
gme_revenue = get_revenue(REVENUE_URLS["GameStop"])

display(tesla_data.head())
display(tesla_revenue.tail())
display(gme_data.head())
display(gme_revenue.tail())

,Date,Close,Volume
0,2010-06-29,1.592667,281494500
1,2010-06-30,1.588667,257806500
2,2010-07-01,1.464000,123282000
3,2010-07-02,1.280000,77097000
4,2010-07-06,1.074000,103003500


,Date,Revenue
48,2021-09-30,13757.0
49,2021-12-31,17719.0
50,2022-03-31,18756.0
51,2022-06-30,16934.0
52,2022-09-30,21454.0


,Date,Close,Volume
0,2002-02-13,1.691667,76216000
1,2002-02-14,1.683250,11021600
2,2002-02-15,1.674834,8389600
3,2002-02-19,1.607504,7410400
4,2002-02-20,1.662210,6892800


,Date,Revenue
57,2019-04-30,1548.0
58,2019-07-31,1286.0
59,2019-10-31,1439.0
60,2020-01-31,2194.0
61,2020-04-30,1021.0


## 2. Share price and revenue over time

In [4]:
def plot_price_and_revenue(stock, revenue, name):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.15,
                        subplot_titles=("Share price (US$)", "Quarterly revenue (US$ millions)"))
    fig.add_trace(go.Scatter(x=stock.Date, y=stock.Close, name="Share price"), row=1, col=1)
    fig.add_trace(go.Bar(x=revenue.Date, y=revenue.Revenue, name="Revenue"), row=2, col=1)
    fig.update_layout(title=name, showlegend=False, height=700)
    fig.show()

plot_price_and_revenue(tesla_data, tesla_revenue, "Tesla")
plot_price_and_revenue(gme_data, gme_revenue, "GameStop")

## 3. Return and risk
I compare total and annualised return, volatility (annualised standard deviation of daily returns) and the worst peak-to-trough fall (maximum drawdown).

In [5]:
def summary(stock, name):
    s = stock.set_index("Date")["Close"]
    daily = s.pct_change().dropna()
    years = (s.index[-1] - s.index[0]).days / 365.25
    return pd.Series({
        "Start": s.index[0].date(), "End": s.index[-1].date(),
        "Total return (%)": round((s.iloc[-1] / s.iloc[0] - 1) * 100, 1),
        "Annualised return (%)": round(((s.iloc[-1] / s.iloc[0]) ** (1 / years) - 1) * 100, 1),
        "Annualised volatility (%)": round(daily.std() * (252 ** 0.5) * 100, 1),
        "Max drawdown (%)": round((s / s.cummax() - 1).min() * 100, 1),
        "Best day (%)": round(daily.max() * 100, 1),
        "Worst day (%)": round(daily.min() * 100, 1),
    }, name=name)

pd.concat([summary(tesla_data, "Tesla"), summary(gme_data, "GameStop")], axis=1)

,Tesla,GameStop
Start,2010-06-29,2002-02-13
End,2021-06-14,2021-06-14
Total return (%),12827.8,3290.7
Annualised return (%),55.8,20.0
Annualised volatility (%),56.6,79.4
Max drawdown (%),-60.6,-93.4
Best day (%),24.4,134.8
Worst day (%),-21.1,-60.0


In [6]:
# Both prices rebased to 100 on the first date both stocks have data
start = max(tesla_data.Date.min(), gme_data.Date.min())
both = pd.merge(tesla_data[tesla_data.Date >= start][["Date", "Close"]],
                gme_data[gme_data.Date >= start][["Date", "Close"]],
                on="Date", suffixes=("_t", "_g"))
both["Tesla"] = both.Close_t / both.Close_t.iloc[0] * 100
both["GameStop"] = both.Close_g / both.Close_g.iloc[0] * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=both.Date, y=both.Tesla, name="Tesla"))
fig.add_trace(go.Scatter(x=both.Date, y=both.GameStop, name="GameStop"))
fig.update_layout(title=f"Growth of 100 invested on {start.date()}", yaxis_title="Value", yaxis_type="log")
fig.show()

## 4. Does revenue move with the share price?
I take the last share price of each quarter and match it to that quarter's revenue.

In [7]:
def quarterly_link(stock, revenue):
    p = stock.groupby(stock.Date.dt.to_period("Q"))["Close"].last()
    r = revenue.groupby(revenue.Date.dt.to_period("Q"))["Revenue"].last()
    q = pd.concat([p, r], axis=1, join="inner")
    q.columns = ["Close", "Revenue"]
    return q

for name, s, r in [("Tesla", tesla_data, tesla_revenue), ("GameStop", gme_data, gme_revenue)]:
    q = quarterly_link(s, r)
    print(f"{name}: correlation of price and revenue levels = {q.corr().iloc[0, 1]:.2f}, "
          f"of quarter-on-quarter changes = {q.pct_change().corr().iloc[0, 1]:.2f}  ({len(q)} quarters)")

Tesla: correlation of price and revenue levels = 0.81, of quarter-on-quarter changes = -0.07  (45 quarters)
GameStop: correlation of price and revenue levels = 0.19, of quarter-on-quarter changes = 0.10  (62 quarters)


In [8]:
# Annual revenue and growth (full years only)
for name, r in [("Tesla", tesla_revenue), ("GameStop", gme_revenue)]:
    g = r.groupby(r.Date.dt.year)["Revenue"].agg(["sum", "count"])
    g = g[g["count"] == 4].drop(columns="count").rename(columns={"sum": "Revenue ($M)"})
    g["Growth (%)"] = (g["Revenue ($M)"].pct_change() * 100).round(1)
    print(name); display(g)

Tesla


,Revenue ($M),Growth (%)
Date,,
2010,116.0,NaN
2011,204.0,75.9
2012,413.0,102.5
2013,2013.0,387.4
2014,3199.0,58.9
2015,4046.0,26.5
2016,7000.0,73.0
2017,11759.0,68.0
2018,21461.0,82.5


GameStop


,Revenue ($M),Growth (%)
Date,,
2005,2134.0,NaN
2006,4682.0,119.4
2007,6532.0,39.5
2008,8180.0,25.2
2009,9047.0,10.6
2010,9305.0,2.9
2011,9665.0,3.9
2012,8904.0,-7.9
2013,8918.0,0.2


## 5. Findings

- **Return:** Tesla grew by 12,828% between June 2010 and June 2021 (about 56% a year). GameStop grew by 3,291% from February 2002 (about 20% a year). The two periods have different lengths, so the annualised figures are the fairer comparison: Tesla compounded almost three times faster.
- **Risk:** GameStop was the riskier stock. Its volatility was 79% against 57% for Tesla, and its worst fall from a peak was 93% against 61%. A 93% fall means the price has to rise about fifteen-fold just to recover.
- **Return for the risk taken:** Tesla earned roughly 1.0 percentage point of annual return per point of volatility. GameStop earned about 0.25.
- **Extreme days:** GameStop's best day was +135%, and its worst was -60%. Tesla's range was +24% to -21%.
- **Revenue:** Tesla's annual revenue grew every year, from 116M in 2010 to 31.5B in 2020, with the slowest year (2019) still at +14.5%. GameStop's revenue peaked at 9.7B in 2011 and fell in six of the eight years from 2012 to 2019, ending 24% below the peak at $7.3B.
- - **Revenue vs price:** Tesla's price and revenue levels were strongly correlated (r = 0.81), because both rose steadily over the decade. Quarter to quarter, though, there was no relationship (r = -0.07, 45 quarters). For GameStop, revenue and price were only weakly related, in levels (r = 0.19) and in quarterly changes (r = 0.10, 62 quarters).
- **Takeaway:** Tesla's long-term share price growth matched its revenue growth, but quarterly revenue did not predict short-term price moves. GameStop's price was largely disconnected from its sales, which fell from 2011 onwards. In both cases, price reflects expectations and events beyond revenue, so revenue alone is a poor short-term guide.

## 6. Limitations
- Revenue data comes from course files that end in 2021, so the comparison stops at 14 June 2021.
- Correlation is not causation, and price also reflects expectations and events beyond revenue.
- This is a learning project, not financial advice.

*Course: IBM Python Project for Data Science (Coursera / IBM Skills Network).*